# Kaggle — Light-CNN + LFCC | ASVspoof 2019 LA

In [ ]:
import subprocess; subprocess.run(['pip', 'install', 'soundfile', 'librosa', '-q'])

In [ ]:

import pandas as pd, os, glob

def find_dataset_root():
    candidates = [
        "/kaggle/input/asvpoof-2019-dataset/LA/LA",
        "/kaggle/input/asvspoof-2019/LA/LA",
        "/kaggle/input/asvpoof2019/LA/LA",
        "/kaggle/input/la-asvspoof2019/LA/LA",
    ]
    # Also scan /kaggle/input/* for LA/LA substructure
    for base in glob.glob("/kaggle/input/*/"):
        for sub in ["LA/LA", "LA"]:
            p = os.path.join(base.rstrip("/"), sub)
            if os.path.exists(os.path.join(p, "ASVspoof2019_LA_train")):
                candidates.insert(0, p)
    for c in candidates:
        if os.path.exists(os.path.join(c, "ASVspoof2019_LA_train")):
            return c
    raise RuntimeError(
        "ASVspoof2019 LA dataset not found.\n"
        "Add the dataset: https://www.kaggle.com/datasets/awsaf49/asvpoof-2019-dataset"
    )

def find_proto_file(proto_dir, split):
    # ASVspoof2019 protocol files have exact names - try them directly
    direct = {
        "train": "ASVspoof2019.LA.cm.train.trn.txt",
        "dev":   "ASVspoof2019.LA.cm.dev.trl.txt",
        "eval":  "ASVspoof2019.LA.cm.eval.trl.txt",
    }
    p = os.path.join(proto_dir, direct[split])
    if os.path.exists(p):
        return p
    # Fallback: glob search (no os.walk, only in proto_dir)
    patterns = {"train": ["*train*trn*.txt", "*train*.txt", "*trn*.txt"],
                "dev":   ["*dev*trl*.txt", "*dev*.txt"],
                "eval":  ["*eval*trl*.txt", "*eval*.txt", "*evl*.txt"]}
    for pat in patterns[split]:
        matches = glob.glob(os.path.join(proto_dir, pat))
        if matches:
            return matches[0]
    return None

def build_manifest(data_root):
    proto_dir = os.path.join(data_root, "ASVspoof2019_LA_cm_protocols")
    if not os.path.exists(proto_dir):
        # search one level up
        for name in ["ASVspoof2019_LA_cm_protocols", "protocols"]:
            p = os.path.join(os.path.dirname(data_root), name)
            if os.path.exists(p):
                proto_dir = p
                break
    print(f"Protocol dir: {proto_dir}")
    print(f"Protocol files: {os.listdir(proto_dir) if os.path.exists(proto_dir) else 'NOT FOUND'}")

    split_to_flac = {
        "train": os.path.join(data_root, "ASVspoof2019_LA_train", "flac"),
        "dev":   os.path.join(data_root, "ASVspoof2019_LA_dev",   "flac"),
        "eval":  os.path.join(data_root, "ASVspoof2019_LA_eval",  "flac"),
    }
    rows = []
    for split, flac_dir in split_to_flac.items():
        proto_file = find_proto_file(proto_dir, split)
        if proto_file is None:
            print(f"WARNING: protocol file not found for {split}")
            continue
        print(f"  [{split}] protocol: {os.path.basename(proto_file)}")
        with open(proto_file, encoding="utf-8") as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) < 5:
                    continue
                spk, aid, _, atk, key = parts[0], parts[1], parts[2], parts[3], parts[4]
                fp = os.path.join(flac_dir, aid + ".flac")
                rows.append({
                    "speaker_id": spk, "audio_id": aid, "attack_id": atk,
                    "key": key, "is_spoof": int(key == "spoof"),
                    "partition": split,          # NOTE: named "partition" to avoid df.split conflict
                    "file_path": fp,
                    "file_exists": os.path.exists(fp),
                })
    df = pd.DataFrame(rows)
    return df

DATA_ROOT = find_dataset_root()
print(f"Dataset root: {DATA_ROOT}")
manifest = build_manifest(DATA_ROOT)
print(f"\nTotal rows: {len(manifest)}")
for split in ["train", "dev", "eval"]:
    sub = manifest[manifest["partition"] == split]   # use ["partition"] not .split
    if len(sub) == 0:
        continue
    bon = len(sub[sub["key"] == "bonafide"])
    spf = len(sub[sub["key"] == "spoof"])
    missing = len(sub[~sub["file_exists"]])
    print(f"  {split}: total={len(sub)} bon={bon} spoof={spf} ratio={spf/bon:.1f}:1 missing={missing}")


In [ ]:
import torch

CFG = {
    "sample_rate":    16000,
    "target_samples": 64000,
    "pre_emphasis":   0.97,
    "vad_top_db":     40,
    "n_fft":          1024,
    "hop_length":     256,
    "n_lfcc":         20,
    "lfcc_frames":    251,
    "batch_size":     128,
    "epochs":         30,
    "lr":             1e-3,
    "lr_min":         1e-6,
    "weight_decay":   1e-4,
    "focal_alpha":    0.75,
    "focal_gamma":    2.0,
    "label_smoothing": 0.05,
    "spec_t_mask":    30,
    "spec_f_mask":    10,
    "dropout":        0.3,
    "seed":           42,
    "device":         "cuda" if torch.cuda.is_available() else "cpu",
    "num_workers":    2,
    "output_dir":     "/kaggle/working",
    "model_name":     "light_cnn_lfcc",
}
torch.manual_seed(CFG["seed"])
import numpy as np; np.random.seed(CFG["seed"])
print(f"Device: {CFG['device']}")
if CFG["device"] == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")


In [ ]:

import numpy as np, soundfile as sf, librosa
import scipy.fftpack as fft_

def load_audio(filepath, target_sr=16000):
    y, sr = sf.read(filepath)
    if y.ndim > 1:
        y = y.mean(axis=1)
    y = y.astype(np.float32)
    if sr != target_sr:
        y = librosa.resample(y, orig_sr=sr, target_sr=target_sr)
    return y, target_sr

def process_waveform(y, is_training=False, cfg=None):
    alpha = cfg["pre_emphasis"] if cfg else 0.97
    target = cfg["target_samples"] if cfg else 64000
    top_db = cfg["vad_top_db"] if cfg else 40
    y = np.concatenate([[y[0]], y[1:] - alpha * y[:-1]])
    intervals = librosa.effects.split(y=y, top_db=top_db)
    if len(intervals) > 0:
        trimmed = np.concatenate([y[s:e] for s, e in intervals])
        if len(trimmed) > 1000:
            y = trimmed
    n = len(y)
    if n >= target:
        start = np.random.randint(0, n - target + 1) if is_training else (n - target) // 2
        y = y[start:start + target]
    else:
        y = np.pad(y, (0, target - n), mode="wrap")
    return y / (np.max(np.abs(y)) + 1e-7)

def extract_lfcc(y, sr=16000, n_fft=1024, hop=256, n_lfcc=20, T=251):
    stft = librosa.stft(y, n_fft=n_fft, hop_length=hop, center=True)
    power = np.abs(stft) ** 2
    n_bins = power.shape[0]
    fbank = np.zeros((n_lfcc, n_bins), dtype=np.float32)
    pts = np.linspace(0, n_bins - 1, n_lfcc + 2, dtype=int)
    for i in range(n_lfcc):
        lo, mid, hi = pts[i], pts[i+1], pts[i+2]
        if mid > lo:
            fbank[i, lo:mid] = np.linspace(0, 1, mid - lo)
        if hi > mid:
            fbank[i, mid:hi] = np.linspace(1, 0, hi - mid)
    energy = np.dot(fbank, power)
    log_e = np.log(np.maximum(energy, 1e-8))
    static = fft_.dct(log_e, type=2, axis=0, norm="ortho")[:n_lfcc]
    delta  = librosa.feature.delta(static, order=1)
    delta2 = librosa.feature.delta(static, order=2)
    feat = np.vstack([static, delta, delta2]).astype(np.float32)
    if feat.shape[1] < T:
        feat = np.pad(feat, ((0,0),(0, T - feat.shape[1])), mode="edge")
    return feat[:, :T]

# Verify preprocessing on one sample
row = manifest[manifest["partition"] == "train"].iloc[0]
y, sr = load_audio(row["file_path"])
y = process_waveform(y, is_training=False, cfg=CFG)
feat = extract_lfcc(y, sr, CFG["n_fft"], CFG["hop_length"], CFG["n_lfcc"], CFG["lfcc_frames"])
print(f"LFCC shape: {feat.shape}  (expected (60, 251))")
assert feat.shape == (60, 251), f"Shape mismatch: {feat.shape}"
print("Preprocessing OK.")


In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

class LFCCDataset(Dataset):
    def __init__(self, df, is_training=False):
        self.df = df.reset_index(drop=True)
        self.is_training = is_training

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        try:
            y, sr = load_audio(row["file_path"])
            y = process_waveform(y, self.is_training, CFG)
            x = extract_lfcc(y, sr, CFG["n_fft"], CFG["hop_length"], CFG["n_lfcc"], CFG["lfcc_frames"])
        except Exception:
            x = np.zeros((60, CFG["lfcc_frames"]), dtype=np.float32)
        if self.is_training:
            if np.random.rand() < 0.5:
                t = np.random.randint(1, CFG["spec_t_mask"] + 1)
                t0 = np.random.randint(0, max(1, CFG["lfcc_frames"] - t))
                x[:, t0:t0+t] = x.mean()
            if np.random.rand() < 0.5:
                f = np.random.randint(1, CFG["spec_f_mask"] + 1)
                f0 = np.random.randint(0, max(1, 60 - f))
                x[f0:f0+f, :] = x.mean()
        return torch.from_numpy(x).unsqueeze(0), torch.tensor(int(row["is_spoof"]), dtype=torch.long)

train_df = manifest[manifest["partition"] == "train"].reset_index(drop=True)
dev_df   = manifest[manifest["partition"] == "dev"].reset_index(drop=True)

labels = train_df["is_spoof"].values
class_counts = np.bincount(labels)
sample_weights = torch.FloatTensor((1.0 / class_counts)[labels])
sampler = WeightedRandomSampler(sample_weights, len(sample_weights), replacement=True)

train_ds = LFCCDataset(train_df, is_training=True)
dev_ds   = LFCCDataset(dev_df,   is_training=False)
train_loader = DataLoader(train_ds, batch_size=CFG["batch_size"], sampler=sampler,
                          num_workers=CFG["num_workers"], pin_memory=True)
dev_loader   = DataLoader(dev_ds, batch_size=CFG["batch_size"]*2, shuffle=False,
                          num_workers=CFG["num_workers"], pin_memory=True)
print(f"Train: {len(train_ds)} | Dev: {len(dev_ds)} | Batches/epoch: {len(train_loader)}")


In [ ]:
import torch.nn as nn

class MFM(nn.Module):
    def forward(self, x):
        a, b = torch.split(x, x.size(1)//2, dim=1)
        return torch.max(a, b)

class LightCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.b1 = nn.Sequential(nn.Conv2d(1, 64, 5, 1, 2),   MFM(), nn.MaxPool2d(2,2))
        self.b2 = nn.Sequential(nn.Conv2d(32, 64, 1), MFM(), nn.BatchNorm2d(32),
                                 nn.Conv2d(32, 96, 3, 1, 1),  MFM(), nn.MaxPool2d(2,2), nn.BatchNorm2d(48))
        self.b3 = nn.Sequential(nn.Conv2d(48, 96, 1), MFM(), nn.BatchNorm2d(48),
                                 nn.Conv2d(48, 128, 3, 1, 1), MFM(), nn.MaxPool2d(2,2))
        self.b4 = nn.Sequential(nn.Conv2d(64, 128, 1), MFM(), nn.BatchNorm2d(64),
                                 nn.Conv2d(64, 64, 3, 1, 1),  MFM(), nn.BatchNorm2d(32))
        self.b5 = nn.Sequential(nn.Conv2d(32, 64, 1), MFM(), nn.BatchNorm2d(32),
                                 nn.Conv2d(32, 64, 3, 1, 1),  MFM(), nn.MaxPool2d(2,2))
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.head = nn.Sequential(nn.Flatten(), nn.Linear(32, 64), nn.ReLU(),
                                   nn.Dropout(CFG["dropout"]), nn.Linear(64, 2))

    def forward(self, x):
        for b in [self.b1, self.b2, self.b3, self.b4, self.b5]:
            x = b(x)
        return self.head(self.pool(x))

device = torch.device(CFG["device"])
model  = LightCNN().to(device)
n = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"LightCNN params: {n:,}")
with torch.no_grad():
    dummy = torch.zeros(2, 1, 60, 251).to(device)
    print(f"Output shape: {model(dummy).shape}  (expected [2, 2])")


In [ ]:

import torch.nn as nn, torch.nn.functional as F
from sklearn.metrics import roc_curve, roc_auc_score

class FocalLoss(nn.Module):
    def __init__(self, alpha=0.75, gamma=2.0, label_smoothing=0.05):
        super().__init__()
        self.alpha, self.gamma, self.ls = alpha, gamma, label_smoothing
    def forward(self, inp, tgt):
        ce = F.cross_entropy(inp, tgt, reduction="none", label_smoothing=self.ls)
        pt = torch.exp(-ce)
        a  = torch.where(tgt == 1, self.alpha, 1.0 - self.alpha)
        return (a * (1 - pt) ** self.gamma * ce).mean()

def compute_eer(y_true, y_score):
    fpr, tpr, _ = roc_curve(y_true, y_score, pos_label=1)
    fnr = 1 - tpr
    idx = np.nanargmin(np.abs(fpr - fnr))
    return float((fpr[idx] + fnr[idx]) / 2)

criterion = FocalLoss(CFG["focal_alpha"], CFG["focal_gamma"], CFG["label_smoothing"])
optimizer = torch.optim.AdamW(model.parameters(), lr=CFG["lr"], weight_decay=CFG["weight_decay"])
scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer, T_0=10, T_mult=2, eta_min=CFG["lr_min"])
scaler = torch.cuda.amp.GradScaler(enabled=(CFG["device"] == "cuda"))
print("Ready to train.")


In [ ]:

import time, json

best_eer, best_auc = float("inf"), 0.0
history = []
save_path = os.path.join(CFG["output_dir"], f"{CFG['model_name']}_best.pth")
os.makedirs(CFG["output_dir"], exist_ok=True)

for epoch in range(1, CFG["epochs"] + 1):
    model.train()
    total_loss, t0 = 0.0, time.time()
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast(enabled=(CFG["device"] == "cuda")):
            loss = criterion(model(xb), yb)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item() * xb.size(0)
    train_loss = total_loss / len(train_ds)
    scheduler.step(epoch)

    model.eval()
    all_probs, all_targets = [], []
    with torch.no_grad():
        for xb, yb in dev_loader:
            with torch.cuda.amp.autocast(enabled=(CFG["device"] == "cuda")):
                logits = model(xb.to(device))
            all_probs.append(torch.softmax(logits, dim=1)[:, 1].cpu().numpy())
            all_targets.append(yb.numpy())
    y_prob = np.concatenate(all_probs)
    y_true = np.concatenate(all_targets)
    eer  = compute_eer(y_true, y_prob)
    auc  = roc_auc_score(y_true, y_prob)
    acc  = ((y_prob >= 0.5) == y_true).mean()

    print(f"Ep {epoch:02d}/{CFG['epochs']} | loss={train_loss:.4f} | EER={eer*100:.2f}% | AUC={auc:.4f} | acc={acc*100:.1f}% | {time.time()-t0:.0f}s")
    history.append({"epoch": epoch, "train_loss": round(train_loss,6),
                    "val_eer": round(eer,6), "val_auc": round(auc,6)})

    if eer < best_eer:
        best_eer, best_auc = eer, auc
        torch.save({
            "epoch": epoch, "model_state_dict": model.state_dict(),
            "eer": best_eer, "auc": best_auc, "cfg": CFG,
        }, save_path)
        print(f"  >>> Best: EER={best_eer*100:.2f}% AUC={best_auc:.4f}")

json.dump(history, open(os.path.join(CFG["output_dir"], f"{CFG['model_name']}_history.json"), "w"), indent=2)
print(f"\nTraining complete. Best EER: {best_eer*100:.2f}% | AUC: {best_auc:.4f}")
print(f"Saved: {save_path}")


In [ ]:

import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
epochs_x = [h["epoch"] for h in history]
axes[0].plot(epochs_x, [h["train_loss"] for h in history], "#3498db", lw=2)
axes[0].set(title="Training Loss", xlabel="Epoch", ylabel="Loss")
eers = [h["val_eer"]*100 for h in history]
axes[1].plot(epochs_x, eers, "#e74c3c", lw=2, marker="o", ms=3)
axes[1].axhline(min(eers), color="gray", ls="--", label=f"Best: {min(eers):.2f}%")
axes[1].set(title="Dev EER (%)", xlabel="Epoch"); axes[1].legend()
aucs = [h["val_auc"] for h in history]
axes[2].plot(epochs_x, aucs, "#2ecc71", lw=2, marker="o", ms=3)
axes[2].axhline(max(aucs), color="gray", ls="--", label=f"Best: {max(aucs):.4f}")
axes[2].set(title="Dev AUC", xlabel="Epoch"); axes[2].legend()
plt.tight_layout()
plt.savefig(os.path.join(CFG["output_dir"], f"{CFG['model_name']}_curve.png"), dpi=150)
plt.show()
